# Clasificación de reseñas hoteleras

**Proyectos de Machine Learning · CENESTUR**  
**Integrantes:** Bryan Velastegui y Evelyn Morocho

El problema consiste en clasificar el texto de una reseña como **negativa (0)** o **positiva (1)** para apoyar la revisión de comentarios de huéspedes. El cuaderno reproduce el flujo de la materia: adquirir datos reales, limpiarlos, separar conjuntos por grupos, entrenar, ajustar parámetros, medir el desempeño y guardar el pipeline que utiliza la API.

Se usa [Andalusian Hotels’ Reviews v3](https://www.kaggle.com/datasets/chizhikchi/andalusian-hotels-reviews-unbalanced/versions/3), publicado por Mariia Chizhikova en Kaggle con licencia declarada CC BY-NC 4.0. Las estrellas son una aproximación al sentimiento.

## 1. Entorno y parámetros

Ejecutar de arriba abajo desde la carpeta del proyecto con las dependencias de `requirements.txt`. Para explorar cambios en la rejilla, dejar `EVALUAR_PRUEBA = False`. Los valores incluidos reproducen el experimento de la aplicación.

In [1]:
import csv
import hashlib
import io
import json
import re
import shutil
import tempfile
import unicodedata
from collections import Counter, defaultdict
from importlib.metadata import version
from pathlib import Path
from statistics import mean, stdev

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import ParameterGrid, StratifiedGroupKFold
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

BASE = Path.cwd()
DATOS = BASE / "datos"
RESULTADOS = BASE / "resultados"
SOURCE = DATOS / "Big_AHR.csv"
DATASET = "chizhikchi/andalusian-hotels-reviews-unbalanced/3"
SHA256_ORIGINAL = "1eac75de3099cee4d7b688444511a1e26b24c225922518d22410351d1cdf5c39"
SEMILLA = 42
C_LOGISTICA = [0.5, 1.0, 2.0]
ALPHA_NAIVE_BAYES = [0.5, 1.0, 2.0]
EVALUAR_PRUEBA = True
EXPORTAR_MODELO = True
DESTINO_MODELO = RESULTADOS / "modelo"
DESTINO_FIGURAS = RESULTADOS / "figuras_modelo"

for carpeta in (DATOS, RESULTADOS, DESTINO_FIGURAS):
    carpeta.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
AZUL = "#234c7c"
TURQUESA = "#0f766e"
NARANJA = "#b9572b"
GRIS = "#687788"


def escribir_json(ruta, datos):
    ruta.write_text(json.dumps(datos, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def sha256(ruta):
    return hashlib.sha256(ruta.read_bytes()).hexdigest()


def huella_csv(filas):
    memoria = io.StringIO(newline="")
    escritor = csv.DictWriter(memoria, fieldnames=list(filas[0]))
    escritor.writeheader()
    escritor.writerows(filas)
    return hashlib.sha256(memoria.getvalue().encode("utf-8")).hexdigest()


def mostrar_figura(figura, nombre):
    figura.tight_layout()
    figura.savefig(DESTINO_FIGURAS / f"{nombre}.png", dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(figura)

## 2. Obtener y comprobar el corpus

Se reutiliza `datos/Big_AHR.csv` si su huella coincide con la versión auditada. Cuando el archivo falta, el cliente oficial de Kaggle descarga la versión 3 con las credenciales locales. Se copia únicamente `Big_AHR.csv`. El archivo balanceado comparte textos con el corpus completo y sus positivos se concentran en cinco estrellas.

In [2]:
if not SOURCE.exists() or sha256(SOURCE) != SHA256_ORIGINAL:
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    with tempfile.TemporaryDirectory(prefix="ahr_kaggle_") as directorio:
        api.dataset_download_files(DATASET, path=directorio, unzip=True, quiet=True)
        descargado = Path(directorio) / SOURCE.name
        if sha256(descargado) != SHA256_ORIGINAL:
            raise ValueError("La huella del archivo descargado difiere de la versión auditada")
        shutil.copyfile(descargado, SOURCE)
    escribir_json(DATOS / "descarga_oficial.json", {
        "dataset": DATASET, "cliente": "kaggle", "version_cliente": version("kaggle"),
        "archivo": SOURCE.name, "sha256": sha256(SOURCE),
    })
    print("Corpus descargado con Kaggle y verificado.")
else:
    print("Archivo local verificado contra el SHA-256 de AHR v3.")

with SOURCE.open(encoding="utf-8-sig", newline="") as archivo:
    originales = list(csv.DictReader(archivo))
print(f"Filas originales: {len(originales):,}")

Archivo local verificado contra el SHA-256 de AHR v3.
Filas originales: 18,172
